In [15]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

df_fe = pd.read_csv("D:/clean_featured.csv")      # <- yahan path badla

y_clf, y_reg = df_fe["emi_eligibility"], df_fe["max_monthly_emi"]
X = df_fe.drop(columns=["emi_eligibility", "max_monthly_emi"])

X_tr, X_tmp, ycl_tr, ycl_tmp, yr_tr, yr_tmp = train_test_split(
    X, y_clf, y_reg, test_size=0.30, stratify=y_clf, random_state=42)
X_val, X_te, ycl_val, ycl_te, yr_val, yr_te = train_test_split(
    X_tmp, ycl_tmp, yr_tmp, test_size=0.50, stratify=ycl_tmp, random_state=42)
print(len(X_tr), len(X_val), len(X_te))

cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
preprocessor.fit(X_tr)

280000 60000 60000


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder().fit(ycl_tr)
ycl_tr_e, ycl_val_e, ycl_te_e = le.transform(ycl_tr), le.transform(ycl_val), le.transform(ycl_te)

SAMPLE = 100_000
idx = X_tr.sample(SAMPLE, random_state=1).index if SAMPLE else X_tr.index
print(le.classes_, len(idx))

['Eligible' 'High_Risk' 'Not_Eligible'] 100000


In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

clf_models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                             n_jobs=-1, eval_metric="mlogloss", random_state=42),
}

def clf_metrics(pipe, X, y):
    pred, proba = pipe.predict(X), pipe.predict_proba(X)
    return {"accuracy": accuracy_score(y, pred),
            "precision": precision_score(y, pred, average="weighted"),
            "recall": recall_score(y, pred, average="weighted"),
            "f1": f1_score(y, pred, average="weighted"),
            "roc_auc": roc_auc_score(y, proba, multi_class="ovr")}

Xs = X_tr.loc[idx]
ys = pd.Series(ycl_tr_e, index=X_tr.index).loc[idx]

mlflow.set_experiment("EMI_Classification")
for name, model in clf_models.items():
    with mlflow.start_run(run_name=name):
        pipe = Pipeline([("prep", clone(preprocessor)), ("model", model)])
        t0 = time.time()
        pipe.fit(Xs, ys)
        mlflow.log_params(model.get_params())
        mlflow.log_param("train_rows", len(Xs))
        m = clf_metrics(pipe, X_val, ycl_val_e)
        mlflow.log_metrics({f"val_{k}": v for k, v in m.items()})
        mlflow.log_metric("train_seconds", time.time() - t0)
        info = mlflow.sklearn.log_model(pipe, name="model", serialization_format="cloudpickle")
        mlflow.set_tag("model_uri", info.model_uri)
        print(name, "done ->", {k: round(v, 4) for k, v in m.items()})

2026/10/24 12:38:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LogisticRegression done -> {'accuracy': 0.9107, 'precision': 0.9095, 'recall': 0.9107, 'f1': 0.9099, 'roc_auc': 0.9765}


2026/10/24 12:38:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest done -> {'accuracy': 0.966, 'precision': 0.9661, 'recall': 0.966, 'f1': 0.9661, 'roc_auc': 0.9961}


2026/10/24 12:38:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost done -> {'accuracy': 0.9819, 'precision': 0.9819, 'recall': 0.9819, 'f1': 0.9819, 'roc_auc': 0.999}


In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

reg_models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, n_jobs=-1, random_state=42),
}

def reg_metrics(pipe, X, y):
    p = pipe.predict(X)
    return {"rmse": float(np.sqrt(mean_squared_error(y, p))),
            "mae": mean_absolute_error(y, p),
            "r2": r2_score(y, p),
            "mape": mean_absolute_percentage_error(y, p)}

yrs = yr_tr.loc[idx]

mlflow.set_experiment("EMI_Regression")
for name, model in reg_models.items():
    with mlflow.start_run(run_name=name):
        pipe = Pipeline([("prep", clone(preprocessor)), ("model", model)])
        t0 = time.time()
        pipe.fit(Xs, yrs)
        mlflow.log_params(model.get_params())
        mlflow.log_param("train_rows", len(Xs))
        m = reg_metrics(pipe, X_val, yr_val)
        mlflow.log_metrics({f"val_{k}": v for k, v in m.items()})
        mlflow.log_metric("train_seconds", time.time() - t0)
        info = mlflow.sklearn.log_model(pipe, name="model", serialization_format="cloudpickle")
        mlflow.set_tag("model_uri", info.model_uri)
        print(name, "done ->", {k: round(v, 4) for k, v in m.items()})

2026/10/24 12:40:19 INFO mlflow.tracking.fluent: Experiment with name 'EMI_Regression' does not exist. Creating a new experiment.
2026/10/24 12:40:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LinearRegression done -> {'rmse': 2330.0842, 'mae': 1144.1078, 'r2': 0.955, 'mape': 0.3304}


2026/10/24 12:40:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest done -> {'rmse': 1077.0664, 'mae': 436.346, 'r2': 0.9904, 'mape': 0.0479}


2026/10/24 12:40:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost done -> {'rmse': 813.7149, 'mae': 388.9846, 'r2': 0.9945, 'mape': 0.0406}


In [23]:
for exp, fname in [("EMI_Classification", "clf_comparison.csv"),
                   ("EMI_Regression", "reg_comparison.csv")]:
    r = mlflow.search_runs(experiment_names=[exp],
                           filter_string="attributes.status = 'FINISHED'")
    cols = ["tags.mlflow.runName"] + [c for c in r.columns if c.startswith("metrics.val_")]
    t = r[cols].rename(columns=lambda c: c.replace("metrics.", "").replace("tags.mlflow.runName", "model"))
    t = t.dropna(axis=1, how="all")                       # khaali columns hatao
    t = t.drop_duplicates(subset="model").reset_index(drop=True)
    print(exp); print(t.round(4).to_string(index=False), "\n")
    t.to_csv(fname, index=False)

EMI_Classification
             model  val_accuracy  val_f1  val_roc_auc  val_precision  val_recall
           XGBoost        0.9819  0.9819       0.9990         0.9819      0.9819
      RandomForest        0.9660  0.9661       0.9961         0.9661      0.9660
LogisticRegression        0.9108  0.9099       0.9765         0.9095      0.9108 

EMI_Regression
           model   val_mae  val_r2  val_mape  val_rmse
         XGBoost  388.9846  0.9945    0.0406  813.7149
    RandomForest  436.3460  0.9904    0.0479 1077.0664
LinearRegression 1144.1078  0.9550    0.3304 2330.0842 



In [24]:
def best_run(experiment, metric, ascending):
    runs = mlflow.search_runs(experiment_names=[experiment],
                              filter_string="attributes.status = 'FINISHED'",
                              order_by=[f"metrics.{metric} {'ASC' if ascending else 'DESC'}"])
    print(experiment, "-> best:", runs.iloc[0]["tags.mlflow.runName"])
    return runs.iloc[0]

best_c = best_run("EMI_Classification", "val_f1", ascending=False)
best_r = best_run("EMI_Regression", "val_rmse", ascending=True)

for run, reg_name, fname in [(best_c, "EMI_Eligibility_Classifier", "clf_best.joblib"),
                             (best_r, "EMI_Max_EMI_Regressor", "reg_best.joblib")]:
    uri = run["tags.model_uri"]
    mlflow.register_model(uri, reg_name)                      # Model Registry
    joblib.dump(mlflow.sklearn.load_model(uri), MODELS_DIR / fname, compress=3)

joblib.dump(le, MODELS_DIR / "label_encoder.joblib")
print("saved:", [p.name for p in MODELS_DIR.iterdir()])

EMI_Classification -> best: XGBoost
EMI_Regression -> best: XGBoost
saved: ['clf_best.joblib', 'label_encoder.joblib', 'reg_best.joblib']


Successfully registered model 'EMI_Eligibility_Classifier'.
Created version '1' of model 'EMI_Eligibility_Classifier'.
Successfully registered model 'EMI_Max_EMI_Regressor'.
Created version '1' of model 'EMI_Max_EMI_Regressor'.


In [25]:
import shutil, os
os.makedirs("data", exist_ok=True)
df_fe.sample(20_000, random_state=1).to_csv("data/sample.csv", index=False)   # EDA aur Admin page ke liye
print(os.listdir("."), os.listdir("models"))

['01_cleaning_eda.ipynb', '02_models_mlflow.ipynb', 'clf_comparison.csv', 'data', 'mlflow.db', 'mlruns', 'models', 'reg_comparison.csv', 'venv'] ['clf_best.joblib', 'label_encoder.joblib', 'reg_best.joblib']


In [26]:
clf = joblib.load(MODELS_DIR / "clf_best.joblib")
reg = joblib.load(MODELS_DIR / "reg_best.joblib")

print("Classification (test):", {k: round(v, 4) for k, v in clf_metrics(clf, X_te, ycl_te_e).items()})
print("Regression (test):    ", {k: round(v, 4) for k, v in reg_metrics(reg, X_te, yr_te).items()})

Classification (test): {'accuracy': 0.9819, 'precision': 0.9819, 'recall': 0.9819, 'f1': 0.9819, 'roc_auc': 0.9991}
Regression (test):     {'rmse': 839.3504, 'mae': 387.9901, 'r2': 0.9941, 'mape': 0.0397}


In [27]:
import os
os.makedirs("src", exist_ok=True)

code = '''import pandas as pd

def add_features(d):
    d = d.copy()
    s = d["monthly_salary"]
    d["total_expenses"] = d[["monthly_rent", "school_fees", "college_fees", "travel_expenses",
                             "groceries_utilities", "other_monthly_expenses"]].sum(axis=1)
    d["requested_emi"] = d["requested_amount"] / d["requested_tenure"]
    d["expense_to_income"] = d["total_expenses"] / s
    d["existing_emi_to_income"] = d["current_emi_amount"] / s
    d["requested_emi_to_income"] = d["requested_emi"] / s
    d["dti_after_loan"] = (d["current_emi_amount"] + d["requested_emi"]) / s
    d["disposable_income"] = s - d["total_expenses"] - d["current_emi_amount"]
    d["affordability_ratio"] = d["requested_emi"] / d["disposable_income"].clip(lower=1)
    d["savings_to_salary"] = (d["bank_balance"] + d["emergency_fund"]) / s
    d["employment_stability"] = d["years_of_employment"] / (d["age"] - 20)
    d["credit_risk_band"] = pd.cut(d["credit_score"], [0, 580, 670, 740, 900],
                                   labels=["Poor", "Fair", "Good", "Excellent"]).astype(str)
    d["credit_x_disposable"] = d["credit_score"] * d["disposable_income"] / 1e5
    d["dependents_x_expense"] = d["dependents"] * d["expense_to_income"]
    return d
'''
open("src/features.py", "w").write(code)
print("src/features.py saved")

src/features.py saved


In [ ]:
from pathlib import Path

ROOT = Path("D:/EMI_PRIDICTION")
(ROOT / "pages").mkdir(exist_ok=True)
(ROOT / "data").mkdir(exist_ok=True)

# sample data (for the Data Explorer and Admin pages)
df_fe.sample(20_000, random_state=1).to_csv(ROOT / "data/sample.csv", index=False)

app = r'''import streamlit as st, pandas as pd, joblib
from src.features import add_features

st.set_page_config(page_title="EMIPredict AI", page_icon="💰", layout="wide")

@st.cache_resource
def load_models():
    return (joblib.load("models/clf_best.joblib"),
            joblib.load("models/reg_best.joblib"),
            joblib.load("models/label_encoder.joblib"))

clf, reg, le = load_models()

RANGES = {  # (amount_min, amount_max, tenure_min, tenure_max) from the training data
    "E-commerce Shopping EMI": (10_000, 200_000, 3, 24),
    "Home Appliances EMI":     (20_000, 300_000, 6, 36),
    "Vehicle EMI":             (80_000, 1_500_000, 12, 84),
    "Personal Loan EMI":       (50_000, 1_000_000, 12, 60),
    "Education EMI":           (50_000, 500_000, 6, 48),
}

st.title("💰 EMIPredict AI")
st.caption("Instant assessment of EMI eligibility and the maximum safe monthly EMI")

with st.form("emi_form"):
    st.subheader("Personal")
    a, b, c, d = st.columns(4)
    age = a.number_input("Age", 25, 60, 32)
    gender = b.selectbox("Gender", ["Male", "Female"])
    marital = c.selectbox("Marital status", ["Single", "Married"])
    education = d.selectbox("Education", ["High School", "Graduate", "Post Graduate", "Professional"])

    st.subheader("Employment & Housing")
    a, b, c, d = st.columns(4)
    salary = a.number_input("Monthly salary (INR)", 15000, 200000, 60000, step=1000)
    emp = b.selectbox("Employment type", ["Private", "Government", "Self-employed"])
    years = c.number_input("Years of employment", 0, 40, 5)
    company = d.selectbox("Company type", ["Small", "Mid-size", "Large", "MNC", "Startup"])
    a, b, c, d = st.columns(4)
    house = a.selectbox("House type", ["Rented", "Own", "Family"])
    rent = b.number_input("Monthly rent", 0, 100000, 12000, step=500)
    family = c.number_input("Family size", 1, 10, 4)
    dependents = d.number_input("Dependents", 0, 9, 1)

    st.subheader("Monthly expenses")
    a, b, c, d, e = st.columns(5)
    school = a.number_input("School fees", 0, 50000, 0, step=500)
    college = b.number_input("College fees", 0, 50000, 0, step=500)
    travel = c.number_input("Travel", 0, 50000, 3000, step=500)
    groceries = d.number_input("Groceries & utilities", 0, 100000, 10000, step=500)
    other = e.number_input("Other expenses", 0, 100000, 4000, step=500)

    st.subheader("Credit & Savings")
    a, b, c, d, e = st.columns(5)
    existing = a.selectbox("Existing loans", ["No", "Yes"])
    cur_emi = b.number_input("Current EMI amount", 0, 100000, 0, step=500)
    credit = c.number_input("Credit score", 300, 850, 700)
    bank = d.number_input("Bank balance", 0, 5000000, 200000, step=5000)
    emerg = e.number_input("Emergency fund", 0, 2000000, 100000, step=5000)

    st.subheader("Loan request")
    a, b, c = st.columns(3)
    scenario = a.selectbox("EMI scenario", ["E-commerce Shopping EMI", "Home Appliances EMI",
                                            "Vehicle EMI", "Personal Loan EMI", "Education EMI"])
    amount = b.number_input("Requested amount (INR)", 10000, 1500000, 200000, step=5000)
    tenure = c.number_input("Tenure (months)", 3, 84, 24)

    go = st.form_submit_button("Assess", type="primary")

if go:
    if years > age - 18:
        st.error("Years of employment cannot exceed your working-age years."); st.stop()
    if dependents >= family:
        st.error("Dependents must be fewer than the family size."); st.stop()

    a_lo, a_hi, t_lo, t_hi = RANGES[scenario]
    if not (a_lo <= amount <= a_hi) or not (t_lo <= tenure <= t_hi):
        st.warning(f"⚠️ For {scenario}, the training data covers amounts from ₹{a_lo:,} to ₹{a_hi:,} "
                   f"and tenures from {t_lo} to {t_hi} months. Predictions outside this range are less reliable.")

    row = pd.DataFrame([{
        "age": age, "gender": gender, "marital_status": marital, "education": education,
        "monthly_salary": salary, "employment_type": emp, "years_of_employment": years,
        "company_type": company, "house_type": house, "monthly_rent": rent,
        "family_size": family, "dependents": dependents, "school_fees": school,
        "college_fees": college, "travel_expenses": travel, "groceries_utilities": groceries,
        "other_monthly_expenses": other, "existing_loans": existing,
        "current_emi_amount": cur_emi, "credit_score": credit, "bank_balance": bank,
        "emergency_fund": emerg, "emi_scenario": scenario,
        "requested_amount": amount, "requested_tenure": tenure}])
    X = add_features(row)

    label = le.inverse_transform(clf.predict(X))[0]
    proba = clf.predict_proba(X)[0]
    max_emi = float(reg.predict(X)[0])
    req_emi = amount / tenure

    c1, c2, c3 = st.columns(3)
    c1.metric("Eligibility", label.replace("_", " "))
    c2.metric("Max safe monthly EMI", f"₹{max_emi:,.0f}")
    c3.metric("Requested EMI", f"₹{req_emi:,.0f}", delta=f"{(max_emi - req_emi):,.0f} margin")

    msg = {"Eligible": "Low risk: the EMI is comfortably affordable.",
           "High_Risk": "Marginal case: the loan is possible, but a higher interest rate may apply.",
           "Not_Eligible": "High risk: this loan request is not recommended."}[label]
    {"Eligible": st.success, "High_Risk": st.warning, "Not_Eligible": st.error}[label](msg)

    st.bar_chart(pd.Series(proba, index=le.classes_, name="probability"))
'''

explorer = r'''import streamlit as st, pandas as pd
st.title("📊 Data Explorer")

df = pd.read_csv("data/sample.csv")
st.caption(f"Sample of {len(df):,} records from the original 400K dataset")

opts = list(df["emi_scenario"].unique())
sc = st.multiselect("EMI scenario", opts, default=opts)
d = df[df["emi_scenario"].isin(sc)]

c1, c2 = st.columns(2)
c1.subheader("Eligibility by scenario")
c1.bar_chart(pd.crosstab(d["emi_scenario"], d["emi_eligibility"]))
c2.subheader("Eligibility by employment type")
c2.bar_chart(pd.crosstab(d["employment_type"], d["emi_eligibility"], normalize="index"))

st.subheader("Credit score vs max EMI")
st.scatter_chart(d.sample(min(3000, len(d)), random_state=1),
                 x="credit_score", y="max_monthly_emi", color="emi_eligibility")

st.subheader("Statistics")
st.dataframe(d.describe().T)
'''

perf = r'''import streamlit as st, pandas as pd
st.title("📈 Model Performance (MLflow)")

clf = pd.read_csv("clf_comparison.csv"); reg = pd.read_csv("reg_comparison.csv")

st.subheader("Classification (validation)")
st.dataframe(clf, hide_index=True)
st.bar_chart(clf.set_index("model")[["val_accuracy", "val_f1", "val_roc_auc"]])

st.subheader("Regression (validation)")
st.dataframe(reg, hide_index=True)
st.bar_chart(reg.set_index("model")[["val_rmse"]])

st.success("Best models: XGBoost Classifier and XGBoost Regressor (registered in the MLflow Model Registry).")
st.caption("The full experiment tracking dashboard is available locally via `mlflow ui`.")
'''

admin = r'''import streamlit as st, pandas as pd, os, shutil
st.title("🛠️ Admin: Data Management (CRUD)")

PATH = "data/records.csv"
if not os.path.exists(PATH):
    shutil.copy("data/sample.csv", PATH)

if "df" not in st.session_state:
    st.session_state.df = pd.read_csv(PATH)

st.caption("Edit: double-click a cell. Add: use the + row below the table. Delete: select a row and press Delete. Then click Save.")
edited = st.data_editor(st.session_state.df, num_rows="dynamic", height=450)

c1, c2, c3 = st.columns([1, 1, 1])
if c1.button("💾 Save changes", type="primary"):
    st.session_state.df = edited.reset_index(drop=True)
    st.session_state.df.to_csv(PATH, index=False)
    st.success(f"Saved. Total records: {len(st.session_state.df):,}")
if c2.button("↩️ Reset to original"):
    shutil.copy("data/sample.csv", PATH)
    st.session_state.df = pd.read_csv(PATH)
    st.rerun()
c3.download_button("⬇️ Download CSV", st.session_state.df.to_csv(index=False), "records.csv")
st.write(f"Total records: {len(st.session_state.df):,}")
'''

(ROOT / "app.py").write_text(app, encoding="utf-8")
(ROOT / "pages/1_Data_Explorer.py").write_text(explorer, encoding="utf-8")
(ROOT / "pages/2_Model_Performance.py").write_text(perf, encoding="utf-8")
(ROOT / "pages/3_Admin_CRUD.py").write_text(admin, encoding="utf-8")
print(sorted(p.name for p in ROOT.iterdir()))

['01_cleaning_eda.ipynb', '02_models_mlflow.ipynb', 'app.py', 'clf_comparison.csv', 'data', 'mlflow.db', 'mlruns', 'models', 'pages', 'reg_comparison.csv', 'src', 'venv']


In [29]:
import importlib.metadata as md
from pathlib import Path

pkgs = ["streamlit", "pandas", "numpy", "scikit-learn", "xgboost", "joblib", "cloudpickle"]
lines = [f"{p}=={md.version(p)}" for p in pkgs]
Path("D:/EMI_PRIDICTION/requirements.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

import os
for f in os.listdir("D:/EMI_PRIDICTION/models"):
    print(f, round(os.path.getsize(f"D:/EMI_PRIDICTION/models/{f}") / 1e6, 1), "MB")          

streamlit==1.66.1
pandas==3.1.0
numpy==2.5.4
scikit-learn==1.9.1
xgboost==3.4.1
joblib==1.6.0
cloudpickle==3.1.2
clf_best.joblib 0.8 MB
label_encoder.joblib 0.0 MB
reg_best.joblib 0.4 MB
